#imports

In [1]:
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split

In [2]:
X,y = make_classification(n_samples=10000, n_features=10,n_informative=3)

In [3]:
X

array([[-0.98236493, -1.90076724, -0.99391126, ...,  0.81216295,
         0.00413557,  0.86881868],
       [-1.43072732, -1.48198024,  1.27391429, ...,  0.2967246 ,
        -1.08144177,  1.40951577],
       [ 0.80880205, -0.02297473, -1.07410574, ...,  0.20033005,
        -0.27949455,  0.57468004],
       ...,
       [ 0.6357916 ,  1.16764759, -0.88453559, ..., -1.0388717 ,
         0.20119251, -1.20553654],
       [ 0.05589003,  0.22238502,  1.51785103, ...,  1.14025454,
         1.72292551, -0.85252435],
       [ 0.34717802, -1.00803672,  1.8860334 , ...,  0.65017817,
         1.49782734, -1.1171739 ]])

#train test split

In [4]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [5]:
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train,y_train)
y_pred = dt.predict(X_test)

print("Decision Tree accuracy",accuracy_score(y_test,y_pred))

Decision Tree accuracy 0.8855


# Bagging

In [6]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=0.5,
    bootstrap=True,
    random_state=42
)

In [7]:
bag.fit(X_train,y_train)

BaggingClassifier(estimator=DecisionTreeClassifier(), max_samples=0.5,
                  n_estimators=500, random_state=42)

In [8]:
y_pred = bag.predict(X_test)

In [9]:
accuracy_score(y_test,y_pred)

0.9275

In [10]:
bag.estimators_samples_[0].shape

(4000,)

In [11]:
bag.estimators_features_[0].shape

(10,)

# Bagging using SVM

In [12]:
bag = BaggingClassifier(
    estimator=SVC(),
    n_estimators=500,
    max_samples=0.25,
    bootstrap=True,
    random_state=42
)

In [13]:
bag.fit(X_train,y_train)
y_pred = bag.predict(X_test)
print("Bagging using SVM",accuracy_score(y_test,y_pred))

Bagging using SVM 0.9055


# Pasting

In [14]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=0.25,
    bootstrap=False, #no duplicate rows
    random_state=42,
    verbose = 1,
    n_jobs=-1
)

In [15]:
bag.fit(X_train,y_train)
y_pred = bag.predict(X_test)
print("Pasting classifier",accuracy_score(y_test,y_pred))

[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done   2 out of   2 | elapsed:   14.1s finished
[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.


Pasting classifier 0.9275


[Parallel(n_jobs=2)]: Done   2 out of   2 | elapsed:    0.3s finished


# Random Subspaces

In [16]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=1.0,
    bootstrap=False,
    max_features=0.5,
    bootstrap_features=True, #duplicate features are allowed
    random_state=42
)

In [17]:
bag.fit(X_train,y_train)
y_pred = bag.predict(X_test)
print("Random Subspaces classifier",accuracy_score(y_test,y_pred))

Random Subspaces classifier 0.9265


In [18]:
bag.estimators_samples_[0].shape

(8000,)

In [19]:
bag.estimators_features_[0].shape

(5,)

# Random Patches

In [20]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=0.25,
    bootstrap=True, #duplicate rows are alloed
    max_features=0.5,
    bootstrap_features=True, #duplicate columns are allowed
    random_state=42
)

In [21]:
bag.fit(X_train,y_train)
y_pred = bag.predict(X_test)
print("Random Patches classifier",accuracy_score(y_test,y_pred))

Random Patches classifier 0.926


# OOB Score

In [22]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=0.25,
    bootstrap=True,
    oob_score=True, #use nonused-data  for testing
    random_state=42
)

In [23]:
bag.fit(X_train,y_train)

BaggingClassifier(estimator=DecisionTreeClassifier(), max_samples=0.25,
                  n_estimators=500, oob_score=True, random_state=42)

In [24]:
bag.oob_score_

0.920375

In [25]:
y_pred = bag.predict(X_test)
print("Accuracy",accuracy_score(y_test,y_pred))

Accuracy 0.9285


# Bagging Tips

- Bagging generally gives better results than Pasting
- Good results come around the 25% to 50% row sampling mark
- Random patches and subspaces should be used while dealing with high dimensional data
- To find the correct hyperparameter values we can do GridSearchCV/RandomSearchCV

# Applying GridSearchCV

In [26]:
from sklearn.model_selection import GridSearchCV

In [27]:
parameters = {
    'n_estimators': [50,100,500],
    'max_samples': [0.1,0.4,0.7,1.0],
    'bootstrap' : [True,False],
    'max_features' : [0.1,0.4,0.7,1.0]
    }

In [28]:
search = GridSearchCV(BaggingClassifier(), parameters, cv=5,n_jobs=True)

In [ ]:
search.fit(X_train,y_train)

In [ ]:
search.best_params_
search.best_score_

In [ ]:
search.best_params_